## STEP-1 : CREATE CONNECTION WITH SUPABASE DATABASE (POSTGRES)

In [0]:
jdbc_url = "---"

properties = {
    "user": "---",  
    "password": "---",
    "driver": "---"
}

## STEP-3 : BUILD MENDELIAN DATA ARCHITECTURE FOR STUDENT (SCD)


### READ RAW DATA

In [0]:
department_df = spark.read.jdbc(
    url=jdbc_url,
    table="department",  
    properties=properties
)

display(department_df)

### BRONZE TABLE (RAW LOAD)

In [0]:
department_df.write.format("delta")\
    .mode("overwrite") \
    .saveAsTable("college_dw.bronze.depatment_raw")

### DISPLAY DATA

In [0]:
display(spark.table("college_dw.bronze.depatment_raw"))

### CLEAN RAW DATA

In [0]:
from pyspark.sql.functions import col

silver_dept_df = department_df.select(
    col("dep_id").alias("department_id"),
    col("dep_name").alias("department_name"),
    col("dep_hod").alias("hod_name"),
    col("dep_hod_mail").alias("hod_email")
).dropDuplicates(["department_id"])

### SILVER TABLE (CLEAN DATA)

In [0]:
silver_dept_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("college_dw.silver.department_clean")

### CREATE TEMP VIEW

In [0]:
silver_dept_df.createOrReplaceTempView("staging_department")

### GOLD TABLE (SCD)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS college_dw.gold.dim_department (
    department_id STRING,
    department_name STRING,
    hod_name STRING,
    hod_email STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
)
""")

### HANDEL UPDATES

In [0]:
spark.sql("""
MERGE INTO college_dw.gold.dim_department AS target
USING staging_department AS source
ON target.department_id = source.department_id AND target.is_current = TRUE

WHEN MATCHED AND (
    target.department_name != source.department_name OR
    target.hod_name != source.hod_name OR
    target.hod_email != source.hod_email
)
THEN UPDATE SET
    target.end_date = current_timestamp(),
    target.is_current = FALSE

WHEN NOT MATCHED
THEN INSERT (
    department_id,
    department_name,
    hod_name,
    hod_email,
    start_date,
    end_date,
    is_current
)
VALUES (
    source.department_id,
    source.department_name,
    source.hod_name,
    source.hod_email,
    current_timestamp(),
    NULL,
    TRUE
)
""")

### HANDEL DELETION

In [0]:
spark.sql("""
UPDATE college_dw.gold.dim_department
SET 
    end_date = current_timestamp(),
    is_current = FALSE
WHERE is_current = TRUE
AND department_id NOT IN (
    SELECT department_id FROM staging_department
)
""")

### DISPLAY RESULT

In [0]:
display(spark.table("college_dw.gold.dim_department"))